In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, 
                           f1_score, accuracy_score, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

class ThreeModelsValidation:
    """
    Validação final dos três modelos com hold-out test sets
    """
    
    def __init__(self):
        # Configurações dos modelos baseadas nos melhores resultados do cross-validation
        self.model_configs = {
            'Model_1': {
                'name': 'Model 1 (Demographic/Perinatal)',
                'algorithm': 'Random Forest',
                'train_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL1TRAIN.csv',
                'test_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL1TEST.csv',
                'expected_auc': 0.570,
                'expected_ci': (0.529, 0.611)
            },
            'Model_2': {
                'name': 'Model 2 (Feeding Practices)',
                'algorithm': 'Logistic Regression',
                'train_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL2TRAIN.csv',
                'test_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL2TEST.csv',
                'expected_auc': 0.598,
                'expected_ci': (0.504, 0.692)
            },
            'Model_3': {
                'name': 'Model 3 (Combined)',
                'algorithm': 'Gradient Boosting',
                'train_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL3TRAIN.csv',
                'test_path': '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL3TEST.csv',
                'expected_auc': 0.571,
                'expected_ci': (0.503, 0.639)
            }
        }
        
        # Variáveis binárias que precisam de correção de tipo
        self.binary_vars_model2_3 = [
            'doou_leite_banco', 'recebeu_leite_banco', 'amamentou_outra_crianca',
            'usou_mamadeira', 'deixou_amamentar_por_outra', 'busca_info_aleitamento',
            'deu_outros_liquidos', 'k15_recebeu', 'k11_amamentou', 'k03_prenatal',
            'usou_utensilios_amamentacao'
        ]
        
        self.results = {}
    
    def create_model_pipeline(self, algorithm, random_state=42):
        """
        Cria pipeline do modelo baseado no algoritmo especificado
        """
        if algorithm == 'Random Forest':
            # Hiperparâmetros otimizados para Random Forest
            model = Pipeline([
                ('scaler', RobustScaler()),
                ('model', RandomForestClassifier(
                    n_estimators=100,
                    max_depth=10,
                    min_samples_split=5,
                    min_samples_leaf=2,
                    random_state=random_state,
                    class_weight='balanced',
                    n_jobs=-1
                ))
            ])
        
        elif algorithm == 'Logistic Regression':
            # Hiperparâmetros otimizados para Logistic Regression (Ridge)
            model = Pipeline([
                ('scaler', RobustScaler()),
                ('model', LogisticRegression(
                    random_state=random_state,
                    max_iter=10000,
                    penalty='l2',  # Ridge
                    C=100.0,
                    class_weight='balanced'
                ))
            ])
        
        elif algorithm == 'Gradient Boosting':
            # Hiperparâmetros otimizados para Gradient Boosting
            model = Pipeline([
                ('scaler', RobustScaler()),
                ('model', GradientBoostingClassifier(
                    n_estimators=100,
                    learning_rate=0.1,
                    max_depth=3,
                    random_state=random_state
                ))
            ])
        
        else:
            raise ValueError(f"Algoritmo não reconhecido: {algorithm}")
        
        return model
    
    def calculate_metrics(self, y_true, y_pred, y_pred_proba):
        """
        Calcula todas as métricas de performance
        """
        metrics = {}
        
        # Métricas básicas
        metrics['auc'] = roc_auc_score(y_true, y_pred_proba)
        metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
        metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
        metrics['accuracy'] = accuracy_score(y_true, y_pred)
        
        # Confusion matrix para specificity e NPV
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
            metrics['npv'] = tn / (tn + fn) if (tn + fn) > 0 else 0
            metrics['ppv'] = tp / (tp + fp) if (tp + fp) > 0 else 0  # Mesmo que precision
        else:
            # Caso especial: só uma classe presente
            if y_true.sum() == 0:  # Só negativos
                metrics['specificity'] = 1.0
                metrics['npv'] = 1.0
                metrics['ppv'] = 0.0
            else:  # Só positivos
                metrics['specificity'] = 0.0
                metrics['npv'] = 0.0
                metrics['ppv'] = 1.0
        
        return metrics
    
    def bootstrap_confidence_intervals(self, y_true, y_pred, y_pred_proba, n_bootstrap=1000):
        """
        Calcula intervalos de confiança 95% usando bootstrap
        """
        np.random.seed(42)
        
        bootstrap_metrics = {
            'auc': [], 'precision': [], 'recall': [], 'f1': [],
            'accuracy': [], 'specificity': [], 'npv': [], 'ppv': []
        }
        
        for i in range(n_bootstrap):
            # Amostragem bootstrap
            indices = np.random.choice(len(y_true), size=len(y_true), replace=True)
            y_true_boot = y_true.iloc[indices] if hasattr(y_true, 'iloc') else y_true[indices]
            y_pred_boot = y_pred[indices]
            y_pred_proba_boot = y_pred_proba[indices]
            
            try:
                metrics_boot = self.calculate_metrics(y_true_boot, y_pred_boot, y_pred_proba_boot)
                
                for metric, value in metrics_boot.items():
                    bootstrap_metrics[metric].append(value)
                    
            except Exception:
                # Em caso de erro, usar métricas originais
                original_metrics = self.calculate_metrics(y_true, y_pred, y_pred_proba)
                for metric, value in original_metrics.items():
                    bootstrap_metrics[metric].append(value)
        
        # Calcular intervalos de confiança
        ci_results = {}
        for metric, values in bootstrap_metrics.items():
            mean_val = np.mean(values)
            ci_lower = np.percentile(values, 2.5)
            ci_upper = np.percentile(values, 97.5)
            
            ci_results[metric] = {
                'mean': mean_val,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper
            }
        
        return ci_results
    
    def validate_single_model(self, model_key):
        """
        Valida um modelo específico
        """
        config = self.model_configs[model_key]
        
        print(f"\n{'='*80}")
        print(f"VALIDAÇÃO FINAL - {config['name'].upper()}")
        print(f"{'='*80}")
        
        # Carregar dados
        df_train = pd.read_csv(config['train_path'])
        df_test = pd.read_csv(config['test_path'])
        
        # Corrigir tipos de dados para modelos 2 e 3
        if model_key in ['Model_2', 'Model_3']:
            for var in self.binary_vars_model2_3:
                if var in df_train.columns:
                    df_train[var] = df_train[var].astype(int)
                if var in df_test.columns:
                    df_test[var] = df_test[var].astype(int)
        
        # Preparar dados
        target_cols = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
        
        y_train = df_train['obeso'].copy()
        X_train = df_train.drop(target_cols + ['id_anon'], axis=1)
        
        y_test = df_test['obeso'].copy()
        X_test = df_test.drop(target_cols + ['id_anon'], axis=1)
        
        print(f"Dados de treinamento:")
        print(f"  - Observações: {len(df_train):,}")
        print(f"  - Features: {X_train.shape[1]}")
        print(f"  - Obesos: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
        
        print(f"\nDados de teste:")
        print(f"  - Observações: {len(df_test):,}")
        print(f"  - Features: {X_test.shape[1]}")
        print(f"  - Obesos: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
        
        # Criar e treinar modelo
        model = self.create_model_pipeline(config['algorithm'])
        
        print(f"\nTreinando modelo final...")
        print(f"  - Algoritmo: {config['algorithm']}")
        print(f"  - Preprocessamento: RobustScaler")
        print(f"  - Balanceamento: class_weight='balanced'")
        
        # Treinar modelo
        model.fit(X_train, y_train)
        
        # Predições no test set
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)
        
        # Calcular métricas básicas
        base_metrics = self.calculate_metrics(y_test, y_pred, y_pred_proba)
        
        print(f"\nResultados no test set (single run):")
        for metric, value in base_metrics.items():
            print(f"  - {metric.upper()}: {value:.3f}")
        
        # Bootstrap para intervalos de confiança
        print(f"\nCalculando intervalos de confiança com bootstrap (n=1,000)...")
        
        ci_results = self.bootstrap_confidence_intervals(y_test, y_pred, y_pred_proba)
        
        # Armazenar resultados
        self.results[model_key] = {
            'config': config,
            'base_metrics': base_metrics,
            'ci_results': ci_results,
            'model': model
        }
        
        return ci_results
    
    def print_results_table(self, model_key):
        """
        Imprime tabela formatada de resultados
        """
        config = self.model_configs[model_key]
        ci_results = self.results[model_key]['ci_results']
        
        print(f"\n{'='*90}")
        print(f"RESULTADOS FINAIS COM IC 95% - {config['name'].upper()}")
        print(f"{'='*90}")
        
        metrics_names = {
            'auc': 'AUC-ROC',
            'precision': 'Precision',
            'recall': 'Recall (Sensitivity)',
            'f1': 'F1-Score',
            'accuracy': 'Accuracy',
            'specificity': 'Specificity',
            'npv': 'NPV',
            'ppv': 'PPV'
        }
        
        print(f"{'Métrica':<25} {'Valor':<10} {'IC 95%':<25} {'Interpretação':<25}")
        print("-" * 90)
        
        for metric, name in metrics_names.items():
            if metric in ci_results:
                result = ci_results[metric]
                mean_val = result['mean']
                ci_lower = result['ci_lower']
                ci_upper = result['ci_upper']
                
                # Interpretação especial para AUC-ROC
                if metric == 'auc':
                    if ci_lower <= 0.5:
                        interpretation = "Inclui acaso (≤0.5)"
                    elif ci_lower > 0.5 and ci_upper < 0.7:
                        interpretation = "Fraco (>0.5, <0.7)"
                    else:
                        interpretation = "Moderado (≥0.7)"
                else:
                    interpretation = ""
                
                ci_str = f"({ci_lower:.3f}-{ci_upper:.3f})"
                print(f"{name:<25} {mean_val:<10.3f} {ci_str:<25} {interpretation:<25}")
    
    def critical_analysis(self, model_key):
        """
        Análise crítica dos resultados
        """
        config = self.model_configs[model_key]
        auc_result = self.results[model_key]['ci_results']['auc']
        precision_result = self.results[model_key]['ci_results']['precision']
        
        print(f"\n{'='*70}")
        print(f"ANÁLISE CRÍTICA - {config['name'].upper()}")
        print(f"{'='*70}")
        
        print(f"Capacidade Preditiva:")
        if auc_result['ci_lower'] <= 0.5:
            print(f"  ❌ AUC IC 95% inclui ≤0.5: NÃO supera o acaso estatisticamente")
            print(f"     Limite inferior: {auc_result['ci_lower']:.3f}")
            print(f"     Diferença do acaso: {auc_result['ci_lower'] - 0.5:.3f}")
        else:
            print(f"  ✓ AUC IC 95% exclui ≤0.5: Supera o acaso estatisticamente")
            print(f"     Limite inferior: {auc_result['ci_lower']:.3f}")
        
        print(f"\nUtilidade Clínica:")
        print(f"  - Precision {precision_result['mean']:.1%}: {100-precision_result['mean']*100:.1f}% falsos positivos")
        
        if precision_result['mean'] < 0.1:
            print(f"  ❌ Precision < 10%: Clinicamente inútil")
        
        print(f"\nComparação com Cross-Validation:")
        print(f"  - CV AUC esperado: {config['expected_auc']:.3f}")
        print(f"  - Hold-out AUC: {auc_result['mean']:.3f}")
        print(f"  - Diferença: {auc_result['mean'] - config['expected_auc']:.3f}")
    
    def run_all_validations(self):
        """
        Executa validação de todos os três modelos
        """
        print("VALIDAÇÃO FINAL DOS TRÊS MODELOS DE PREDIÇÃO DE OBESIDADE INFANTIL")
        print("="*80)
        print("Objetivo: Confirmar que nenhum modelo supera o acaso estatisticamente")
        
        # Validar cada modelo
        for model_key in ['Model_1', 'Model_2', 'Model_3']:
            self.validate_single_model(model_key)
            self.print_results_table(model_key)
            self.critical_analysis(model_key)
        
        # Resumo comparativo final
        self.comparative_summary()
    
    def comparative_summary(self):
        """
        Resumo comparativo final dos três modelos
        """
        print(f"\n{'='*80}")
        print("RESUMO COMPARATIVO FINAL")
        print(f"{'='*80}")
        
        print(f"{'Modelo':<30} {'Algoritmo':<20} {'AUC-ROC':<15} {'IC 95%':<20} {'Supera Acaso?'}")
        print("-" * 90)
        
        for model_key in ['Model_1', 'Model_2', 'Model_3']:
            config = self.results[model_key]['config']
            auc_result = self.results[model_key]['ci_results']['auc']
            
            auc_mean = auc_result['mean']
            ci_str = f"({auc_result['ci_lower']:.3f}-{auc_result['ci_upper']:.3f})"
            supera_acaso = "NÃO" if auc_result['ci_lower'] <= 0.5 else "SIM"
            
            print(f"{config['name']:<30} {config['algorithm']:<20} {auc_mean:<15.3f} {ci_str:<20} {supera_acaso}")
        
        print(f"\n{'='*80}")
        print("CONCLUSÃO CIENTÍFICA GERAL")
        print(f"={'='*80}")
        print("EVIDÊNCIA ROBUSTA DE LIMITAÇÕES PREDITIVAS:")
        print("  - TODOS os modelos falharam em superar o acaso estatisticamente")
        print("  - Fatores dos primeiros 24 meses inadequados para predição de obesidade")
        print("  - Validação hold-out confirma resultados negativos do cross-validation")
        print("  - Precision consistentemente <5% indica alta taxa de falsos positivos")
        
        print(f"\nImplicações para Pesquisa e Política:")
        print("  - Paradigma de predição precoce deve ser questionado")
        print("  - Foco deve migrar para fatores proximais modificáveis")
        print("  - Intervenções baseadas nestes preditores seriam ineficazes")
        print("  - Resultados negativos são cientificamente valiosos")

# Executar validação completa
if __name__ == "__main__":
    validator = ThreeModelsValidation()
    validator.run_all_validations()

VALIDAÇÃO FINAL DOS TRÊS MODELOS DE PREDIÇÃO DE OBESIDADE INFANTIL
Objetivo: Confirmar que nenhum modelo supera o acaso estatisticamente

VALIDAÇÃO FINAL - MODEL 1 (DEMOGRAPHIC/PERINATAL)
Dados de treinamento:
  - Observações: 6,588
  - Features: 27
  - Obesos: 183 (2.8%)

Dados de teste:
  - Observações: 1,648
  - Features: 27
  - Obesos: 46 (2.8%)

Treinando modelo final...
  - Algoritmo: Random Forest
  - Preprocessamento: RobustScaler
  - Balanceamento: class_weight='balanced'

Resultados no test set (single run):
  - AUC: 0.598
  - PRECISION: 0.000
  - RECALL: 0.000
  - F1: 0.000
  - ACCURACY: 0.971
  - SPECIFICITY: 0.999
  - NPV: 0.972
  - PPV: 0.000

Calculando intervalos de confiança com bootstrap (n=1,000)...

RESULTADOS FINAIS COM IC 95% - MODEL 1 (DEMOGRAPHIC/PERINATAL)
Métrica                   Valor      IC 95%                    Interpretação            
------------------------------------------------------------------------------------------
AUC-ROC                   0.